# 02 · Silver transform
Each cleaning function in `food_pipeline/silver.py` fixes one issue found in `00_explore.ipynb`.
This notebook applies them one at a time on Bronze, so you can see what each one changes.

Step 1 only cleans in memory. Data quality rules, deduplication and writing the Silver table come in the next steps.

In [1]:
from pathlib import Path
from pyspark.sql import functions as F
from food_pipeline.spark_session import get_spark
from food_pipeline import silver

ROOT = Path.cwd().parent
spark = get_spark()
BRONZE = (ROOT / "lakehouse" / "bronze" / "survey_raw").as_posix()

bronze = spark.read.format("delta").load(BRONZE)
print("bronze rows:", bronze.count())

bronze rows: 388


## 1 · `trim_strings`
Fixes `"Negative "` (trailing space). Brackets make spaces visible: before vs after.

In [2]:
def bracketed(df, column):
    return df.groupBy(F.concat(F.lit("["), F.col(column), F.lit("]")).alias(column)).count()

trimmed = silver.trim_strings(bronze)
bracketed(bronze, "feedback").show()
bracketed(trimmed, "feedback").show()

+-----------+-----+
|   feedback|count|
+-----------+-----+
|[Negative ]|   71|
| [Positive]|  317|
+-----------+-----+



+----------+-----+
|  feedback|count|
+----------+-----+
|[Positive]|  317|
|[Negative]|   71|
+----------+-----+



## 2 · `standardise_categories`
`Self Employeed` → `Self Employed`, `House wife` → `Homemaker`.

In [3]:
silver.standardise_categories(trimmed).groupBy("occupation").count().orderBy(F.desc("count")).show()

+-------------+-----+
|   occupation|count|
+-------------+-----+
|      Student|  207|
|     Employee|  118|
|Self Employed|   54|
|    Homemaker|    9|
+-------------+-----+



## 3 · `cast_types`
`age`, `family_size` → int; `latitude`, `longitude` → double; `pin_code` stays a string (it's an ID).
A value that can't be cast becomes **null**, so we count nulls to prove nothing was lost.

In [4]:
typed = silver.cast_types(trimmed)
typed.select("age", "family_size", "latitude", "longitude", "pin_code").printSchema()
typed.select([F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_nulls")
              for c in silver.COLUMN_TYPES]).show()
typed.select(F.min("age"), F.max("age"), F.min("family_size"), F.max("family_size")).show()

root
 |-- age: integer (nullable = true)
 |-- family_size: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- pin_code: string (nullable = true)



+---------+-----------------+--------------+---------------+
|age_nulls|family_size_nulls|latitude_nulls|longitude_nulls|
+---------+-----------------+--------------+---------------+
|        0|                0|             0|              0|
+---------+-----------------+--------------+---------------+



+--------+--------+----------------+----------------+
|min(age)|max(age)|min(family_size)|max(family_size)|
+--------+--------+----------------+----------------+
|      18|      33|               1|               6|
+--------+--------+----------------+----------------+



## 4 · `map_income`
Text bands get a sortable number plus numeric bounds. An unknown band would map to null.

In [5]:
(silver.map_income(trimmed)
    .groupBy("monthly_income", "income_band_order", "income_min_inr", "income_max_inr").count()
    .orderBy("income_band_order").show(truncate=False))

+---------------+-----------------+--------------+--------------+-----+
|monthly_income |income_band_order|income_min_inr|income_max_inr|count|
+---------------+-----------------+--------------+--------------+-----+
|No Income      |0                |0             |0             |187  |
|Below Rs.10000 |1                |1             |10000         |25   |
|10001 to 25000 |2                |10001         |25000         |45   |
|25001 to 50000 |3                |25001         |50000         |69   |
|More than 50000|4                |50001         |NULL          |62   |
+---------------+-----------------+--------------+--------------+-----+



## 5 · `clean_outcome`
`output` (Yes/No text) → `ordered_again` (boolean). Assumption, documented in the README: `Output` means *the respondent would order again*.

In [6]:
silver.clean_outcome(trimmed).groupBy("ordered_again").count().show()

+-------------+-----+
|ordered_again|count|
+-------------+-----+
|         true|  301|
|        false|   87|
+-------------+-----+



## All steps together: `clean`
Row count must not change: cleaning fixes values, it never adds or drops rows.

In [7]:
cleaned = silver.clean(bronze)
print("rows in:", bronze.count(), "rows out:", cleaned.count())
cleaned.printSchema()
cleaned.drop("_ingest_ts", "_source_file").show(5, truncate=False)

rows in: 388 rows out: 388
root
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- monthly_income: string (nullable = true)
 |-- educational_qualifications: string (nullable = true)
 |-- family_size: integer (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- pin_code: string (nullable = true)
 |-- feedback: string (nullable = true)
 |-- raw_extra: string (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- income_band_order: integer (nullable = true)
 |-- income_min_inr: integer (nullable = true)
 |-- income_max_inr: integer (nullable = true)
 |-- ordered_again: boolean (nullable = true)



+---+------+--------------+----------+--------------+--------------------------+-----------+-------------+--------+---------+--------+--------+---------+---------+-----------------+--------------+--------------+-------------+
|age|gender|marital_status|occupation|monthly_income|educational_qualifications|family_size|customer_type|latitude|longitude|pin_code|feedback|raw_extra|_batch_id|income_band_order|income_min_inr|income_max_inr|ordered_again|
+---+------+--------------+----------+--------------+--------------------------+-----------+-------------+--------+---------+--------+--------+---------+---------+-----------------+--------------+--------------+-------------+
|20 |Female|Single        |Student   |No Income     |Post Graduate             |4          |Frequent     |12.9766 |77.5993  |560001  |Positive|Yes      |batch_01 |0                |0             |0             |true         |
|24 |Female|Single        |Student   |Below Rs.10000|Graduate                  |3          |Regu

# Data quality rules and quarantine
Every rule in `food_pipeline/quality.py` is a condition that must be **true** for a row to be trusted.
Rows that break any rule go to **quarantine** with a `failed_rules` list; they are kept for inspection, never silently deleted.

In [8]:
from food_pipeline import quality

for name, _ in quality.dq_rules():
    print(name)

age_18_to_80
family_size_1_to_15
pin_code_bengaluru
latitude_in_bengaluru
longitude_in_bengaluru
income_band_known
ordered_again_known
raw_extra_matches_output
gender_allowed
marital_status_allowed
occupation_allowed
customer_type_allowed
feedback_allowed


## Run the rules on the real data
Profiling found no invalid values, so we expect **0 failures** on every rule. That result is worth reporting: it is evidence, not an assumption.

In [9]:
counts = quality.failure_counts(cleaned)
for rule, failed in counts.items():
    print(f"{rule:28} {failed}")

valid, quarantine = quality.split_valid(cleaned)
print("\nvalid:", valid.count(), "quarantined:", quarantine.count())
print("raw_extra dropped from valid:", "raw_extra" not in valid.columns)

age_18_to_80                 0
family_size_1_to_15          0
pin_code_bengaluru           0
latitude_in_bengaluru        0
longitude_in_bengaluru       0
income_band_known            0
ordered_again_known          0
raw_extra_matches_output     0
gender_allowed               0
marital_status_allowed       0
occupation_allowed           0
customer_type_allowed        0
feedback_allowed             0



valid: 388 quarantined: 0
raw_extra dropped from valid: True


## Prove the rules work: inject bad rows
Zero failures only means something if the rules *can* fail. We copy one clean row and break it in different ways.

In [10]:
template = cleaned.limit(1)
bad_rows = (
    template.withColumn("age", F.lit(150))
    .unionByName(template.withColumn("pin_code", F.lit("400001")).withColumn("latitude", F.lit(19.07)))
    .unionByName(template.withColumn("feedback", F.lit(None).cast("string")))
    .unionByName(template.withColumn("income_band_order", F.lit(None).cast("int")))
    .unionByName(template.withColumn("raw_extra", F.lit("No")))
)
_, bad_quarantine = quality.split_valid(cleaned.unionByName(bad_rows))
bad_quarantine.select("age", "pin_code", "latitude", "feedback", "income_band_order", "raw_extra", "failed_rules") \
    .show(truncate=False)

+---+--------+--------+--------+-----------------+---------+-------------------------------------------+
|age|pin_code|latitude|feedback|income_band_order|raw_extra|failed_rules                               |
+---+--------+--------+--------+-----------------+---------+-------------------------------------------+
|150|560001  |12.9766 |Positive|0                |Yes      |[age_18_to_80]                             |
|20 |400001  |19.07   |Positive|0                |Yes      |[pin_code_bengaluru, latitude_in_bengaluru]|
|20 |560001  |12.9766 |NULL    |0                |Yes      |[feedback_allowed]                         |
|20 |560001  |12.9766 |Positive|NULL             |Yes      |[income_band_known]                        |
|20 |560001  |12.9766 |Positive|0                |No       |[raw_extra_matches_output]                 |
+---+--------+--------+--------+-----------------+---------+-------------------------------------------+



## Write the quarantine table
`replaceWhere` replaces only the rows of the batches being processed, so rerunning a batch never duplicates quarantine rows.
With this data the table is created empty, which still documents its schema for the quality report.

In [11]:
QUARANTINE = (ROOT / "lakehouse" / "quarantine" / "survey").as_posix()
batch_ids = [r["_batch_id"] for r in bronze.select("_batch_id").distinct().collect()]

quality.write_quarantine(quarantine, QUARANTINE, batch_ids)
quality.write_quarantine(quarantine, QUARANTINE, batch_ids)   # rerun on purpose: still no duplicates

q = spark.read.format("delta").load(QUARANTINE)
print("quarantine rows:", q.count(), "| batches processed:", sorted(batch_ids))
q.printSchema()

quarantine rows: 0 | batches processed: ['batch_01', 'batch_02']
root
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- monthly_income: string (nullable = true)
 |-- educational_qualifications: string (nullable = true)
 |-- family_size: integer (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- pin_code: string (nullable = true)
 |-- feedback: string (nullable = true)
 |-- raw_extra: string (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- income_band_order: integer (nullable = true)
 |-- income_min_inr: integer (nullable = true)
 |-- income_max_inr: integer (nullable = true)
 |-- ordered_again: boolean (nullable = true)
 |-- failed_rules: array (nullable = true)
 |    |-- 

# Deduplicate and MERGE into Silver
The data has no customer ID, so `record_hash` (SHA-256 of the cleaned answers) is the key.
Identical answers give an identical hash, which is how duplicates are found.

**Trade-off (see `docs/decisions.md`):** two different people who gave exactly the same answers are merged into one row.

In [12]:
hashed = silver.add_record_hash(valid)
print("rows:", hashed.count(), "| distinct record_hash:", hashed.select("record_hash").distinct().count())

# one duplicated answer set, to see what a duplicate looks like
dup_hash = hashed.groupBy("record_hash").count().orderBy(F.desc("count")).first()
print("most repeated answer set appears", dup_hash["count"], "times")
hashed.filter(F.col("record_hash") == dup_hash["record_hash"]) \
    .select("age", "gender", "occupation", "pin_code", "feedback", "_batch_id", F.substring("record_hash", 1, 12).alias("hash_prefix")) \
    .show(truncate=False)

rows: 388 | distinct record_hash: 285


most repeated answer set appears 6 times


+---+------+----------+--------+--------+---------+------------+
|age|gender|occupation|pin_code|feedback|_batch_id|hash_prefix |
+---+------+----------+--------+--------+---------+------------+
|23 |Female|Student   |560024  |Positive|batch_01 |ce402f3a3039|
|23 |Female|Student   |560024  |Positive|batch_01 |ce402f3a3039|
|23 |Female|Student   |560024  |Positive|batch_01 |ce402f3a3039|
|23 |Female|Student   |560024  |Positive|batch_01 |ce402f3a3039|
|23 |Female|Student   |560024  |Positive|batch_01 |ce402f3a3039|
|23 |Female|Student   |560024  |Positive|batch_02 |ce402f3a3039|
+---+------+----------+--------+--------+---------+------------+



## Load Bronze → Silver, one batch at a time
For each batch: clean → DQ split (quarantine written) → hash → dedupe within the batch → MERGE.
The MERGE inserts only hashes Silver doesn't have yet, so duplicates *across* batches are skipped too.

This loop is pipeline orchestration; it moves into `run_pipeline.py` in Phase 5.

In [13]:
from delta.tables import DeltaTable

SILVER = (ROOT / "lakehouse" / "silver" / "survey").as_posix()

def silver_count():
    return spark.read.format("delta").load(SILVER).count() if DeltaTable.isDeltaTable(spark, SILVER) else 0

def load_batch_to_silver(batch_id):
    batch = bronze.filter(F.col("_batch_id") == batch_id)
    batch_valid, batch_quarantine = quality.split_valid(silver.clean(batch))
    quality.write_quarantine(batch_quarantine, QUARANTINE, [batch_id])
    hashed_batch = silver.add_record_hash(batch_valid)
    unique_batch = silver.dedupe(hashed_batch)

    before = silver_count()
    silver.merge_into_silver(spark, unique_batch, SILVER)
    after = silver_count()

    n_valid, n_unique = hashed_batch.count(), unique_batch.count()
    return {"batch": batch_id, "bronze_rows": batch.count(), "quarantined": batch_quarantine.count(),
            "dupes_in_batch": n_valid - n_unique, "already_in_silver": n_unique - (after - before),
            "inserted": after - before, "silver_total": after}

batch_ids = sorted(r["_batch_id"] for r in bronze.select("_batch_id").distinct().collect())
stats = [load_batch_to_silver(b) for b in batch_ids]
spark.createDataFrame(stats).select(list(stats[0])).show()

+--------+-----------+-----------+--------------+-----------------+--------+------------+
|   batch|bronze_rows|quarantined|dupes_in_batch|already_in_silver|inserted|silver_total|
+--------+-----------+-----------+--------------+-----------------+--------+------------+
|batch_01|        200|          0|            23|                0|     177|         177|
|batch_02|        188|          0|            68|               12|     108|         285|
+--------+-----------+-----------+--------------+-----------------+--------+------------+



## Prove idempotency: run every batch again
Every hash already exists in Silver, so nothing is inserted and the count stays the same.

In [14]:
rerun = [load_batch_to_silver(b) for b in batch_ids]
spark.createDataFrame(rerun).select(list(rerun[0])).show()
print("silver rows after rerun:", silver_count())

+--------+-----------+-----------+--------------+-----------------+--------+------------+
|   batch|bronze_rows|quarantined|dupes_in_batch|already_in_silver|inserted|silver_total|
+--------+-----------+-----------+--------------+-----------------+--------+------------+
|batch_01|        200|          0|            23|              177|       0|         285|
|batch_02|        188|          0|            68|              120|       0|         285|
+--------+-----------+-----------+--------------+-----------------+--------+------------+



silver rows after rerun: 285


## Reconciliation and history
Every Bronze row must be accounted for: **Bronze = Silver + duplicates removed + quarantined**.

In [15]:
silver_df = spark.read.format("delta").load(SILVER)
n_bronze, n_silver = bronze.count(), silver_df.count()
n_quarantine = spark.read.format("delta").load(QUARANTINE).count()
n_dupes = sum(s["dupes_in_batch"] + s["already_in_silver"] for s in stats)
print(f"bronze {n_bronze} = silver {n_silver} + duplicates {n_dupes} + quarantined {n_quarantine}:",
      n_bronze == n_silver + n_dupes + n_quarantine)
print("record_hash unique in silver:", silver_df.select("record_hash").distinct().count() == n_silver)

spark.sql(f"DESCRIBE HISTORY delta.`{SILVER}`") \
    .select("version", "operation", "operationMetrics.numOutputRows", "operationMetrics.numTargetRowsInserted") \
    .orderBy("version").show(truncate=False)

bronze 388 = silver 285 + duplicates 103 + quarantined 0: True


record_hash unique in silver: True


+-------+---------+-------------+---------------------+
|version|operation|numOutputRows|numTargetRowsInserted|
+-------+---------+-------------+---------------------+
|0      |WRITE    |177          |NULL                 |
|1      |MERGE    |108          |108                  |
+-------+---------+-------------+---------------------+

